# Introduction

The ENVI objects are here: `/data/beroukhim1/youyun/plgg/data/sc_integration/ENVI_results`

Goal: identify pathways specifically enriched in `Myeloid 1` versus `Myeloid 2` in PA spatial imputed expression.


In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import anndata as ad
import glob
import re
import seaborn as sns
from pathlib import Path
from gprofiler import GProfiler

# local vs eristwo
if os.path.expanduser('~') in ["/Users/youyun", "/Users/youyunzheng"]:
    # in a local mac, the home directory is usuaully at '/Users/[username]'
    workdir = os.path.expanduser('~')+"/Documents/HMS/PhD/beroukhimlab/dfci_mount/"
elif os.path.expanduser('~') == "/home/yz762":
    # on the new server, the home directory is /home/yz762
    workdir = "/mnt/storage/dept/medonc/beroukhim/"
else:
    # in eristwo, the home directory is usuaully at '/home/unix/[username]'
    workdir = "/data/beroukhim1/"

# Load Data

## Load the AnnData object


In [ ]:
# List all files matching the pattern
sn_data = sorted(glob.glob(workdir + "youyun/plgg/data/sc_integration/ENVI_results/*astrocytoma_sn_envi.h5ad"))
spatial_data = sorted(glob.glob(workdir + "youyun/plgg/data/sc_integration/ENVI_results/*astrocytoma_spatial_envi.h5ad"))

sn_by_sample = {
    re.sub('_sn_envi.h5ad$', '', Path(x).name): x
    for x in sn_data
}
spatial_by_sample = {
    re.sub('_spatial_envi.h5ad$', '', Path(x).name): x
    for x in spatial_data
}

sample_ids = sorted(set(sn_by_sample.keys()) & set(spatial_by_sample.keys()))
print(f"Matched samples: {len(sample_ids)}")
print(sample_ids)

# Pairwise Myeloid DE and Enrichment


In [ ]:
output_dir = Path(workdir) / "youyun/plgg/code/sc_integration/ENVI/2_spatial_imputation/PA"
output_dir.mkdir(parents=True, exist_ok=True)

target_groups = ["Myeloid 1", "Myeloid 2"]
min_cells_per_group = 20
logfc_cutoff = 0.25
fdr_cutoff = 0.05
min_genes_for_enrichment = 10
enrichment_sources = ["GO:BP", "REAC", "KEGG", "WP"]

pairwise_deg_results = []
pairwise_enrich_results = []
skipped_samples = []

gp = GProfiler(return_dataframe=True)

def build_impute_adata(spatial_adata):
    imputation = spatial_adata.obsm["imputation"]
    if isinstance(imputation, pd.DataFrame):
        out = ad.AnnData(
            X=imputation.to_numpy(),
            obs=spatial_adata.obs.copy(),
            var=pd.DataFrame(index=imputation.columns.astype(str))
        )
    else:
        out = ad.AnnData(X=imputation, obs=spatial_adata.obs.copy())
        if out.n_vars == spatial_adata.n_vars:
            out.var_names = spatial_adata.var_names.copy()
    out.var_names_make_unique()
    return out

for sample_id in sample_ids:
    sample_name = sample_id.replace("_", " ")
    print(f"\nProcessing {sample_name}")
    spatial_adata = sc.read(spatial_by_sample[sample_id])
    impute_adata = build_impute_adata(spatial_adata)

    myeloid_mask = impute_adata.obs["Fine.Cell.Type.UMAP"].isin(target_groups)
    myeloid_adata = impute_adata[myeloid_mask].copy()
    cell_counts = myeloid_adata.obs["Fine.Cell.Type.UMAP"].value_counts()
    count_m1 = int(cell_counts.get("Myeloid 1", 0))
    count_m2 = int(cell_counts.get("Myeloid 2", 0))
    print(f"Myeloid 1 cells: {count_m1}, Myeloid 2 cells: {count_m2}")

    if (count_m1 < min_cells_per_group) or (count_m2 < min_cells_per_group):
        skipped_samples.append(
            {
                "sample": sample_name,
                "myeloid_1_n": count_m1,
                "myeloid_2_n": count_m2,
                "reason": f"requires >= {min_cells_per_group} cells in each group"
            }
        )
        continue

    for group, reference in [("Myeloid 1", "Myeloid 2"), ("Myeloid 2", "Myeloid 1")]:
        contrast_name = f"{group} > {reference}"

        sc.tl.rank_genes_groups(
            myeloid_adata,
            groupby="Fine.Cell.Type.UMAP",
            groups=[group],
            reference=reference,
            method="wilcoxon"
        )
        df_deg = sc.get.rank_genes_groups_df(myeloid_adata, group=group)
        df_deg["sample"] = sample_name
        df_deg["group"] = group
        df_deg["reference"] = reference
        df_deg["contrast"] = contrast_name
        pairwise_deg_results.append(df_deg)

        sig_up = df_deg[
            (df_deg["logfoldchanges"] > logfc_cutoff) &
            (df_deg["pvals_adj"] < fdr_cutoff)
        ].copy()
        sig_up = sig_up.sort_values("pvals_adj")
        up_genes = sig_up["names"].dropna().astype(str).drop_duplicates().tolist()

        print(f"{contrast_name}: {len(up_genes)} significant up genes")

        if len(up_genes) < min_genes_for_enrichment:
            continue

        enrich = gp.profile(
            organism="hsapiens",
            query=up_genes,
            sources=enrichment_sources
        )
        if enrich is None or enrich.empty:
            continue

        enrich["sample"] = sample_name
        enrich["group"] = group
        enrich["reference"] = reference
        enrich["contrast"] = contrast_name
        enrich["n_input_genes"] = len(up_genes)
        pairwise_enrich_results.append(enrich)

if pairwise_deg_results:
    pairwise_deg_results = pd.concat(pairwise_deg_results, ignore_index=True)
else:
    pairwise_deg_results = pd.DataFrame(
        columns=["names", "scores", "logfoldchanges", "pvals", "pvals_adj", "sample", "group", "reference", "contrast"]
    )

if pairwise_enrich_results:
    pairwise_enrich_results = pd.concat(pairwise_enrich_results, ignore_index=True)
else:
    pairwise_enrich_results = pd.DataFrame(
        columns=["source", "native", "name", "p_value", "description", "sample", "group", "reference", "contrast", "n_input_genes"]
    )

pairwise_deg_path = output_dir / "myeloid_pairwise_deg_results.csv"
pairwise_enrich_path = output_dir / "myeloid_pairwise_enrich_results.csv"
pairwise_deg_results.to_csv(pairwise_deg_path, index=False)
pairwise_enrich_results.to_csv(pairwise_enrich_path, index=False)

if skipped_samples:
    skipped_df = pd.DataFrame(skipped_samples)
    skipped_df

print(f"Saved DEG results: {pairwise_deg_path}")
print(f"Saved enrichment results: {pairwise_enrich_path}")
print(f"DE rows: {pairwise_deg_results.shape[0]}, Enrichment rows: {pairwise_enrich_results.shape[0]}")

# Marker Gene Consensus Across Samples


In [ ]:
if pairwise_deg_results.empty:
    gene_consensus = pd.DataFrame(columns=["contrast", "names", "n_samples", "median_logfc", "median_fdr", "sample_fraction"])
    print("No pairwise DEG results available for consensus marker analysis.")
else:
    deg_sig = pairwise_deg_results[
        (pairwise_deg_results["logfoldchanges"] > logfc_cutoff) &
        (pairwise_deg_results["pvals_adj"] < fdr_cutoff)
    ].copy()

    gene_consensus = (
        deg_sig
        .groupby(["contrast", "names"], as_index=False)
        .agg(
            n_samples=("sample", "nunique"),
            median_logfc=("logfoldchanges", "median"),
            median_fdr=("pvals_adj", "median")
        )
    )

    n_samples_total = pairwise_deg_results["sample"].nunique()
    gene_consensus["sample_fraction"] = gene_consensus["n_samples"] / max(1, n_samples_total)
    gene_consensus = gene_consensus.sort_values(
        ["contrast", "n_samples", "median_logfc"],
        ascending=[True, False, False]
    )

    for contrast_name in sorted(gene_consensus["contrast"].unique()):
        print(f"\nTop consensus genes for {contrast_name}")
        print(
            gene_consensus[gene_consensus["contrast"] == contrast_name]
            .head(20)
            .to_string(index=False)
        )

gene_consensus_path = output_dir / "myeloid_pairwise_gene_consensus.csv"
gene_consensus.to_csv(gene_consensus_path, index=False)
print(f"Saved gene consensus: {gene_consensus_path}")

# Pathway Enrichment Consensus and Specificity


In [ ]:
if pairwise_enrich_results.empty:
    term_summary = pd.DataFrame(
        columns=["contrast", "source", "name", "description", "n_samples", "best_p", "median_neg_log10_p", "mean_precision", "max_intersection", "rank_score"]
    )
    term_wide = pd.DataFrame(
        columns=["source", "name", "description", "delta_n_samples_m1_minus_m2", "delta_score_m1_minus_m2"]
    )
    print("No enrichment results available for pathway summary.")
else:
    enrich_sig = pairwise_enrich_results[pairwise_enrich_results["p_value"] < 0.05].copy()
    enrich_sig["neg_log10_p"] = -np.log10(np.clip(enrich_sig["p_value"], 1e-300, None))

    term_summary = (
        enrich_sig
        .groupby(["contrast", "source", "name", "description"], as_index=False)
        .agg(
            n_samples=("sample", "nunique"),
            best_p=("p_value", "min"),
            median_neg_log10_p=("neg_log10_p", "median"),
            mean_precision=("precision", "mean"),
            max_intersection=("intersection_size", "max")
        )
    )
    term_summary["rank_score"] = term_summary["n_samples"] * term_summary["median_neg_log10_p"]
    term_summary = term_summary.sort_values(
        ["contrast", "n_samples", "median_neg_log10_p"],
        ascending=[True, False, False]
    )

    term_wide = term_summary.pivot_table(
        index=["source", "name", "description"],
        columns="contrast",
        values=["n_samples", "median_neg_log10_p"],
        fill_value=0
    )
    term_wide.columns = [f"{left}__{right}" for left, right in term_wide.columns]
    term_wide = term_wide.reset_index()

    for col in [
        "n_samples__Myeloid 1 > Myeloid 2",
        "n_samples__Myeloid 2 > Myeloid 1",
        "median_neg_log10_p__Myeloid 1 > Myeloid 2",
        "median_neg_log10_p__Myeloid 2 > Myeloid 1"
    ]:
        if col not in term_wide.columns:
            term_wide[col] = 0

    term_wide["delta_n_samples_m1_minus_m2"] = (
        term_wide["n_samples__Myeloid 1 > Myeloid 2"] -
        term_wide["n_samples__Myeloid 2 > Myeloid 1"]
    )
    term_wide["delta_score_m1_minus_m2"] = (
        term_wide["median_neg_log10_p__Myeloid 1 > Myeloid 2"] -
        term_wide["median_neg_log10_p__Myeloid 2 > Myeloid 1"]
    )

    my1_specific = term_wide.sort_values(
        ["delta_n_samples_m1_minus_m2", "delta_score_m1_minus_m2"],
        ascending=[False, False]
    )
    my2_specific = term_wide.sort_values(
        ["delta_n_samples_m1_minus_m2", "delta_score_m1_minus_m2"],
        ascending=[True, True]
    )

    print("\nMost Myeloid 1-specific terms")
    print(
        my1_specific[
            (my1_specific["delta_n_samples_m1_minus_m2"] > 0) |
            (my1_specific["delta_score_m1_minus_m2"] > 0)
        ][[
            "source", "name", "delta_n_samples_m1_minus_m2", "delta_score_m1_minus_m2"
        ]]
        .head(20)
        .to_string(index=False)
    )

    print("\nMost Myeloid 2-specific terms")
    print(
        my2_specific[
            (my2_specific["delta_n_samples_m1_minus_m2"] < 0) |
            (my2_specific["delta_score_m1_minus_m2"] < 0)
        ][[
            "source", "name", "delta_n_samples_m1_minus_m2", "delta_score_m1_minus_m2"
        ]]
        .head(20)
        .to_string(index=False)
    )

term_summary_path = output_dir / "myeloid_pairwise_enrich_summary.csv"
term_summary.to_csv(term_summary_path, index=False)
print(f"Saved enrichment summary: {term_summary_path}")

specificity_path = output_dir / "myeloid_pairwise_enrich_specificity.csv"
term_wide.to_csv(specificity_path, index=False)
print(f"Saved pathway specificity table: {specificity_path}")

# Visual Summaries


In [ ]:
if term_summary.empty:
    plot_terms = pd.DataFrame(columns=["contrast", "source", "name", "term_label", "n_samples", "median_neg_log10_p", "rank_score"])
    print("No pathway terms to plot.")
else:
    plot_terms = (
        term_summary[term_summary["n_samples"] >= 2]
        .sort_values(["contrast", "rank_score"], ascending=[True, False])
        .groupby("contrast", as_index=False)
        .head(15)
        .copy()
    )

    plot_terms["term_label"] = plot_terms["name"].str.slice(0, 70)

    contrasts = sorted(plot_terms["contrast"].unique())
    fig, axes = plt.subplots(1, max(1, len(contrasts)), figsize=(8 * max(1, len(contrasts)), 8))
    if len(contrasts) == 1:
        axes = [axes]

    for ax, contrast_name in zip(axes, contrasts):
        sub = plot_terms[plot_terms["contrast"] == contrast_name].copy()
        sub = sub.sort_values("median_neg_log10_p", ascending=True)
        sns.scatterplot(
            data=sub,
            x="median_neg_log10_p",
            y="term_label",
            hue="source",
            size="n_samples",
            sizes=(40, 220),
            ax=ax
        )
        ax.set_title(contrast_name)
        ax.set_xlabel("Median -log10(p)")
        ax.set_ylabel("")
        ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
if plot_terms.empty:
    print("No pathway terms available for heatmap.")
else:
    top_heat_terms = plot_terms[["contrast", "source", "name"]].drop_duplicates()
    heat_df = pairwise_enrich_results.merge(
        top_heat_terms,
        on=["contrast", "source", "name"],
        how="inner"
    ).copy()
    heat_df["neg_log10_p"] = -np.log10(np.clip(heat_df["p_value"], 1e-300, None))
    heat_df["term_label"] = (
        heat_df["contrast"] + " | " + heat_df["source"] + " | " + heat_df["name"].str.slice(0, 45)
    )

    heat_mat = heat_df.pivot_table(
        index="sample",
        columns="term_label",
        values="neg_log10_p",
        aggfunc="max",
        fill_value=0
    )

    if heat_mat.shape[1] > 0:
        col_order = heat_mat.mean(axis=0).sort_values(ascending=False).index
        heat_mat = heat_mat.loc[:, col_order]
        plt.figure(figsize=(max(10, 0.35 * heat_mat.shape[1]), 4))
        sns.heatmap(heat_mat, cmap="mako", linewidths=0.2, linecolor="white")
        plt.title("Per-sample significance of top pathway terms")
        plt.xlabel("")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()